# Label the sampled Reddit comments

Input: `reddit_text_1000.csv`. Output: `reddit_text_1000_labeled.csv`.
Original columns are preserved. `comment_toxicity` is a JSON array of toxicity probabilities (0–1): element i scores element i in `comments`. `num_toxic_comments` counts scores >= 0.5.

Run cells in order. The first run downloads model weights. Comments are scored individually, and truncated to 128 tokens, matching the counter. Scores are model estimates, not human labels.


In [1]:
%pip install -q transformers torch pandas numpy tqdm

Note: you may need to restart the kernel to use updated packages.


In [7]:
import json, os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ---------------- CONFIG ----------------
# Works when launched from the repository root or the Reddit directory.
DATA_DIR = Path.cwd() if Path("reddit_text_10000.csv").exists() else Path.cwd() / "Reddit"
INPUT_CSV = DATA_DIR / "reddit_text_10000.csv"
OUTPUT_CSV = DATA_DIR / "reddit_text_10000_labeled.csv"
COMMENTS_COL = "comments"
MODEL_KEY = "toxic-bert"
THRESHOLD = 0.5
MAX_LEN = 128
BATCH_SIZE = 32
CHUNK_ROWS = 100
OVERWRITE = True

# (hugging face id, how logits become probabilities, fallback index of the "toxic" class)
MODELS = {
    "toxic-bert":              ("unitary/toxic-bert",                                       "sigmoid", 0),
    "multilingual-distilbert": ("citizenlab/distilbert-base-multilingual-cased-toxicity",   "softmax", 0),
    "roberta":                 ("s-nlp/roberta_toxicity_classifier",                        "softmax", 1),
    "xlmr-large":              ("textdetox/xlmr-large-toxicity-classifier-v2",              "softmax", 1),
}

assert os.path.abspath(INPUT_CSV) != os.path.abspath(OUTPUT_CSV), "OUTPUT_CSV must be a different file"
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
use_amp = device.type == "cuda"
print("Device:", device)
if device.type == "cpu":
    print("WARNING: no GPU. This will be slow on large files; test on a small CSV first.")

Device: mps


## Load the toxicity model

In [8]:
hf_id, activation, fallback_idx = MODELS[MODEL_KEY]
tokenizer = AutoTokenizer.from_pretrained(hf_id)
model = AutoModelForSequenceClassification.from_pretrained(hf_id).to(device).eval()

# find which output is the "toxic" class (by label name, else the fallback index)
id2label = {int(k): str(v).lower() for k, v in model.config.id2label.items()}
tox_idx = next((i for i, name in id2label.items() if name == "toxic"), fallback_idx)
print("labels:", id2label, "-> using index", tox_idx, "as the toxic score")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

labels: {0: 'toxic', 1: 'severe_toxic', 2: 'obscene', 3: 'threat', 4: 'insult', 5: 'identity_hate'} -> using index 0 as the toxic score


In [9]:
def extract_comments(cell):
    # Strict parsing preserves one-to-one alignment; never silently skip a comment.
    comments = json.loads(cell)
    if not isinstance(comments, list) or not all(isinstance(t, str) and t.strip() for t in comments):
        raise ValueError("Expected a JSON array of nonempty comment strings")
    return comments


def score_texts(texts):
    # returns P(toxic) for each text, in the original order
    if not texts:
        return np.zeros(0)
    order = np.argsort([len(t) for t in texts])          # sort by length -> much less padding -> faster
    sorted_texts = [texts[i] for i in order]
    out = []
    for i in range(0, len(sorted_texts), BATCH_SIZE):
        enc = tokenizer(sorted_texts[i:i + BATCH_SIZE], truncation=True, max_length=MAX_LEN,
                        padding=True, return_tensors="pt").to(device)
        with torch.inference_mode(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(**enc).logits.float()
        probs = torch.sigmoid(logits) if activation == "sigmoid" else torch.softmax(logits, dim=-1)
        out.append(probs[:, tox_idx].cpu().numpy())
    sorted_probs = np.concatenate(out)
    probs = np.empty_like(sorted_probs)
    probs[order] = sorted_probs
    return probs

## Label and export

In [10]:
def process_file():
    if OUTPUT_CSV.exists() and not OVERWRITE:
        raise FileExistsError(OUTPUT_CSV)
    temporary = OUTPUT_CSV.with_suffix(".csv.tmp")
    total_posts = total_comments = total_toxic = 0
    first = True
    try:
        reader = pd.read_csv(INPUT_CSV, dtype=str, keep_default_na=False, chunksize=CHUNK_ROWS)
        for chunk in tqdm(reader, desc="Labeling posts"):
            comments_by_post = [extract_comments(value) for value in chunk[COMMENTS_COL]]
            texts = [text for comments in comments_by_post for text in comments]
            probabilities = score_texts(texts)
            if len(probabilities) != len(texts) or not np.isfinite(probabilities).all():
                raise ValueError("Model returned missing or invalid scores")
            scores_by_post = []
            offset = 0
            for comments in comments_by_post:
                scores = probabilities[offset:offset + len(comments)].tolist()
                assert len(scores) == len(comments)
                scores_by_post.append(scores)
                offset += len(comments)
            chunk["comment_toxicity"] = [json.dumps(scores, allow_nan=False) for scores in scores_by_post]
            chunk["num_toxic_comments"] = [sum(score >= THRESHOLD for score in scores) for scores in scores_by_post]
            chunk.to_csv(temporary, mode="w" if first else "a", header=first, index=False)
            first = False
            total_posts += len(chunk)
            total_comments += len(texts)
            total_toxic += int(chunk["num_toxic_comments"].sum())
        if first:
            raise ValueError("Input CSV contains no posts")
        temporary.replace(OUTPUT_CSV)
    except BaseException:
        temporary.unlink(missing_ok=True)
        raise
    print(f"Saved {total_posts:,} posts and {total_comments:,} comment scores to {OUTPUT_CSV}")
    print(f"{total_toxic:,} comments scored at or above {THRESHOLD}.")

process_file()


Labeling posts: 0it [00:00, ?it/s]

Saved 10,000 posts and 156,315 comment scores to /Users/jeremyzhang/Desktop/Datathon-2026/Reddit/reddit_text_10000_labeled.csv
9,192 comments scored at or above 0.5.


## Verify every score aligns with its comment

In [12]:
labeled = pd.read_csv(OUTPUT_CSV, dtype=str, keep_default_na=False)
original = pd.read_csv(INPUT_CSV, dtype=str, keep_default_na=False)
pd.testing.assert_frame_equal(labeled[original.columns], original)
for comments, scores, count in zip(labeled["comments"], labeled["comment_toxicity"], labeled["num_toxic_comments"]):
    comments, scores = json.loads(comments), json.loads(scores)
    assert len(comments) == len(scores)
    assert all(0 <= score <= 1 for score in scores)
    assert int(count) == sum(score >= THRESHOLD for score in scores)
print(f"Verified all {len(labeled):,} posts: original fields preserved and scores aligned.")
labeled.head()


Verified all 10,000 posts: original fields preserved and scores aligned.


,title,description,comments,comment_toxicity,num_toxic_comments
0,So what are auctioned houses going for compare...,Thinking about the Waikato. We are in the mark...,"[""You can generally look up auction results an...","[0.0006122507620602846, 0.000645090127363801, ...",0
1,Hows the Architecture Industry,I'm Canadian but grew up and work in Hong Kong...,"[""I’m an engineer in AEC industry, there is wo...","[0.0008482924313284457, 0.0006549436948262155,...",0
2,Small House Storage Ideas/Solutions?,"By pure luck, my husband and I were able to pu...","[""Under-bed storage boxes are game changers an...","[0.024000633507966995, 0.0007206632290035486, ...",0
3,Worked at a StartUp it Sucked,Warning: ask lots of questions before you join...,"[""Sorry for your experience..."", ""Zero reading...","[0.0006490582600235939, 0.0057919640094041824,...",0
4,"[HIRING] Backend, Infra & Agentic AI Engineers...",I work with a curated talent network that matc...,"[""do you hire remote ? I've over 3 YOE with Te...","[0.0028175250627100468, 0.0006366739980876446,...",0
